# Sliding-Window Pairwise Electrode Correlation

Compute a time-resolved `n_ch x n_ch` zero-lag Pearson correlation matrix for each of BLT, P1, and P2 (500 ms gap only), on a shared time axis, then compare.

- Signal per condition: trial-averaged evoked.
- Window: `WINDOW_MS` ms, step `STEP_MS` ms.
- Output per condition: `(n_windows, n_ch, n_ch)` + window-center times.

## A. Setup & config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from utils import load_eeg_data
from connectivity import (
    select_p2_500ms,
    sliding_corr_from_epochs,
    summary_metrics,
)

SUBJECT = 5
TMIN, TMAX = None, None       # native epoch window (must match across files)
WINDOW_MS = 200
STEP_MS = 50
FREQ_BAND = None              # e.g. 'alpha' / 'beta' for Cell G

## B. Load the three conditions on a common time axis

In [ ]:
blt = load_eeg_data('BLT', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p1  = load_eeg_data('P1',  SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2_all = load_eeg_data('P2', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2 = select_p2_500ms(p2_all)

assert np.allclose(blt.times, p1.times), 'BLT and P1 time axes differ'
assert np.allclose(p1.times, p2.times), 'P1 and P2 time axes differ'

# Channel-set alignment: intersect across files so the NxN matrices line up.
common_ch = [c for c in blt.ch_names if c in p1.ch_names and c in p2.ch_names]
for ep in (blt, p1, p2):
    ep.pick_channels(common_ch, ordered=True)

print(f'n_channels (common): {len(common_ch)}')
print(f'n_times: {len(blt.times)}   sfreq: {blt.info["sfreq"]} Hz')
print(f'trials BLT/P1/P2_500ms: {len(blt)}/{len(p1)}/{len(p2)}')

## C. Compute sliding-window correlation tensors

In [ ]:
conds = {'BLT': blt, 'P1': p1, 'P2_500ms': p2}
results = {}
for name, ep in conds.items():
    corr, centers, ch_names = sliding_corr_from_epochs(ep, WINDOW_MS, STEP_MS)
    results[name] = {'corr': corr, 'centers': centers, 'ch_names': ch_names}
    print(f'{name}: corr {corr.shape}  centers [{centers[0]:.3f}s .. {centers[-1]:.3f}s]')

# Sanity: all conditions share the same window grid.
ref_centers = results['BLT']['centers']
for name in ('P1', 'P2_500ms'):
    assert np.allclose(results[name]['centers'], ref_centers), f'{name} window centers differ'

## D. Time-course summary per condition

`mean_abs_r` = average `|r|` across unique channel pairs; `density(0.5)` = fraction of pairs with `|r| > 0.5`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name, res in results.items():
    m = summary_metrics(res['corr'])
    axes[0].plot(res['centers'], m['mean_abs_r'], label=name)
    axes[1].plot(res['centers'], m['density'], label=name)
axes[0].set_ylabel('mean |r|')
axes[1].set_ylabel('density (|r|>0.5)')
axes[1].set_xlabel('time (s)')
for ax in axes:
    ax.axvline(0, color='k', lw=0.5, alpha=0.4)
    ax.legend(loc='best', fontsize=9)
fig.suptitle(f'Sliding-window connectivity (window={WINDOW_MS} ms, step={STEP_MS} ms)')
plt.tight_layout()
plt.show()

## E. Heatmap grid at key time points

Pick a few window indices and show the full `n_ch x n_ch` matrix for each condition.

In [ ]:
centers = results['BLT']['centers']
n_windows = len(centers)
# Pick 5 evenly spaced windows across the epoch.
pick_idx = np.linspace(0, n_windows - 1, 5, dtype=int)
pick_times = centers[pick_idx]

cond_names = list(results.keys())
fig, axes = plt.subplots(
    len(cond_names), len(pick_idx),
    figsize=(3.0 * len(pick_idx), 3.0 * len(cond_names)),
    squeeze=False,
)
for i, name in enumerate(cond_names):
    corr = results[name]['corr']
    for j, w in enumerate(pick_idx):
        ax = axes[i, j]
        im = ax.imshow(corr[w], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
        if j == 0:
            ax.set_ylabel(name, fontsize=11)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Pearson r')
fig.suptitle('Correlation matrices at selected windows')
plt.show()

## F. Condition-difference heatmaps

P1 - BLT isolates the effect of adding the auditory cue (with a fixed 500 ms gap). P2_500ms - P1 isolates paradigm/block differences when the audio->tactile interval is matched.

In [ ]:
diffs = {
    'P1 - BLT': results['P1']['corr'] - results['BLT']['corr'],
    'P2_500ms - P1': results['P2_500ms']['corr'] - results['P1']['corr'],
}
# Symmetric color range per diff for fair reading.
fig, axes = plt.subplots(
    len(diffs), len(pick_idx),
    figsize=(3.0 * len(pick_idx), 3.0 * len(diffs)),
    squeeze=False,
)
for i, (name, d) in enumerate(diffs.items()):
    vmax = np.nanpercentile(np.abs(d), 99)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    for j, w in enumerate(pick_idx):
        ax = axes[i, j]
        im = ax.imshow(d[w], vmin=-vmax, vmax=vmax, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
        if j == 0:
            ax.set_ylabel(name, fontsize=11)
    fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.7, label=f'\u0394r (vmax={vmax:.2f})')
fig.suptitle('Condition differences in correlation structure')
plt.show()

## G. (Optional) Per-band repeat

Re-run B-F with `FREQ_BAND = 'alpha'` or `'beta'` to check whether the dynamics are band-specific.